In [ ]:
import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from connection import get_data_from_api

In [1]:
from nltk.corpus import stopwords

# Mengambil daftar stopwords bahasa Indonesia
stop_words = stopwords.words('indonesian')

# Menampilkan daftar stopwords
print(stop_words)

['ada', 'adalah', 'adanya', 'adapun', 'agak', 'agaknya', 'agar', 'akan', 'akankah', 'akhir', 'akhiri', 'akhirnya', 'aku', 'akulah', 'amat', 'amatlah', 'anda', 'andalah', 'antar', 'antara', 'antaranya', 'apa', 'apaan', 'apabila', 'apakah', 'apalagi', 'apatah', 'artinya', 'asal', 'asalkan', 'atas', 'atau', 'ataukah', 'ataupun', 'awal', 'awalnya', 'bagai', 'bagaikan', 'bagaimana', 'bagaimanakah', 'bagaimanapun', 'bagi', 'bagian', 'bahkan', 'bahwa', 'bahwasanya', 'baik', 'bakal', 'bakalan', 'balik', 'banyak', 'bapak', 'baru', 'bawah', 'beberapa', 'begini', 'beginian', 'beginikah', 'beginilah', 'begitu', 'begitukah', 'begitulah', 'begitupun', 'bekerja', 'belakang', 'belakangan', 'belum', 'belumlah', 'benar', 'benarkah', 'benarlah', 'berada', 'berakhir', 'berakhirlah', 'berakhirnya', 'berapa', 'berapakah', 'berapalah', 'berapapun', 'berarti', 'berawal', 'berbagai', 'berdatangan', 'beri', 'berikan', 'berikut', 'berikutnya', 'berjumlah', 'berkali-kali', 'berkata', 'berkehendak', 'berkeinginan'

In [ ]:
# Mengambil data dari API
profiles_rating = get_data_from_api('profiles-rating')
product_data = get_data_from_api('product-data')

# Mengambil data dari database menggunakan query
dt_profiles_rating_df = pd.DataFrame(profiles_rating)
dt_product_df = pd.DataFrame(product_data)

In [ ]:
# Menggantikan nilai-nilai yang kosong dengan nilai 0
dt_profiles_rating_df['review_id'] = dt_profiles_rating_df['review_id'].fillna(0).astype(int)
dt_profiles_rating_df['product_id'] = dt_profiles_rating_df['product_id'].fillna(0).astype(int)
dt_profiles_rating_df['rating'] = dt_profiles_rating_df['rating'].fillna(0).astype(float)

# Menggantikan nilai-nilai yang kosong dengan nilai 0
dt_product_df['rating'] = dt_product_df['rating'].fillna(0).astype(float)

In [ ]:
# Setup stemmer and stop words
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

def preprocess_text(text):
    if text is None:
        return text
    # Case folding
    print(f"Original text: {text}")
    text = text.lower()
    print(f"Lowercased text: {text}")
    
    # Remove punctuation (including underscores)
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    print(f"Punctuation removed text: {text}")
    
    # Remove extra whitespaces
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    
    # Tokenizing
    tokens = word_tokenize(text)
    print(f"Tokenized text: {tokens}")
    
    # Stopwords removing
    tokens = [word for word in tokens if word not in stop_words]
    print(f"Stopwords removed text: {tokens}")
    
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    text = ' '.join(tokens)
    print(f"Stemmed text: {text}")
    
    return text


In [ ]:
text_colums_user = [
    'gender', 'skin_type_face', 'hair_issue', 'skin_type_body', 
    'allergy_history', 'preferred_products', 'avoided_products', 
    'specific_needs'
]

print("Preprocessing dt_profiles_rating_df:")
for column in text_colums_user:
    display(f"\nProcessing column: {column}")
    dt_profiles_rating_df[column] = dt_profiles_rating_df[column].apply(preprocess_text)
    display(f"Preprocessed column '{column}':\n{dt_profiles_rating_df[column].head()}")

print("\nPreprocessed dt_profiles_rating_df:")
display(dt_profiles_rating_df)

In [ ]:
text_colums_content = ['gender', 'skin_type_face', 'hair_issue', 'skin_type_body']

print("\nPreprocessing dt_product_df:")
for column in text_colums_content:
    display(f"\nProcessing column: {column}")
    dt_product_df[column] = dt_product_df[column].apply(preprocess_text)
    display(f"Preprocessed column '{column}':\n{dt_product_df[column].head()}")

print("\nPreprocessed dt_product_df:")
display(dt_product_df)